In [ ]:
import matplotlib.pyplot as plt

from link_prediction.config import (
    RESULTS_DIR,
    load_experiment_config,
    load_methods_config,
)
from link_prediction.statistical_analysis import (
    load_benchmark_fold_metrics,
    run_confirmatory_analysis,
)

In [ ]:
benchmark_name = "standard"

experiment_config = (
    load_experiment_config()
)

methods_config = (
    load_methods_config()
)

statistics_config = (
    experiment_config[
        "statistics"
    ]
)

method_ids = list(
    statistics_config[
        "confirmatory_methods"
    ]
)

metrics = tuple(
    statistics_config[
        "metrics"
    ]
)

alpha = float(
    statistics_config[
        "alpha"
    ]
)

method_names = {
    method_id:
        methods_config[
            "methods"
        ][
            method_id
        ][
            "name"
        ]
    for method_id in method_ids
}

figure_directory = (
    RESULTS_DIR
    / "figures"
    / benchmark_name
)

figure_directory.mkdir(
    parents=True,
    exist_ok=True,
)

fold_metrics = (
    load_benchmark_fold_metrics(
        benchmark_name=
            benchmark_name,
    )
)

statistical_results = (
    run_confirmatory_analysis(
        fold_metrics=
            fold_metrics,
        method_ids=
            method_ids,
        metrics=
            metrics,
        alpha=
            alpha,
        benchmark_name=
            benchmark_name,
    )
)

network_metrics = (
    statistical_results[
        "network_metrics"
    ]
)

friedman = statistical_results[
    "friedman"
]

mean_ranks = statistical_results[
    "mean_ranks"
]

pairwise = statistical_results[
    "pairwise"
]

In [ ]:
friedman[
    [
        "metric",
        "network_count",
        "method_count",
        "statistic",
        "p_value",
        "alpha",
        "reject_null",
    ]
]

In [ ]:
rank_table = (
    mean_ranks.copy()
)

rank_table[
    "method"
] = rank_table[
    "method_id"
].map(
    method_names
)

rank_table[
    [
        "metric",
        "method_id",
        "method",
        "mean_rank",
    ]
].sort_values(
    [
        "metric",
        "mean_rank",
        "method_id",
    ]
).reset_index(
    drop=True
)

In [ ]:
for metric in metrics:
    metric_ranks = (
        rank_table[
            rank_table[
                "metric"
            ]
            == metric
        ]
        .sort_values(
            [
                "mean_rank",
                "method",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )

    figure, axis = plt.subplots(
        figsize=(
            7,
            4,
        )
    )

    axis.barh(
        metric_ranks[
            "method"
        ],
        metric_ranks[
            "mean_rank"
        ],
        color="#4472C4",
    )

    axis.set_xlabel(
        "Mean rank across networks"
    )

    axis.set_ylabel(
        "Method"
    )

    axis.set_xlim(
        1.0,
        float(
            len(method_ids)
        ),
    )

    axis.grid(
        axis="x",
        alpha=0.25,
    )

    figure.tight_layout()

    figure.savefig(
        figure_directory
        / (
            f"{metric}_"
            "confirmatory_mean_ranks.png"
        ),
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        figure_directory
        / (
            f"{metric}_"
            "confirmatory_mean_ranks.pdf"
        ),
        bbox_inches="tight",
    )

    plt.show()

In [ ]:
pairwise_table = (
    pairwise.copy()
)

if not pairwise_table.empty:
    pairwise_table[
        "first_method_name"
    ] = pairwise_table[
        "first_method"
    ].map(
        method_names
    )

    pairwise_table[
        "second_method_name"
    ] = pairwise_table[
        "second_method"
    ].map(
        method_names
    )

pairwise_table[
    [
        "metric",
        "first_method_name",
        "second_method_name",
        "network_count",
        "p_value",
        "adjusted_p_value",
        "reject_null",
        "median_difference",
        "rank_biserial",
        "first_wins",
        "ties",
        "second_wins",
    ]
] if not pairwise_table.empty else pairwise_table

In [ ]:
network_metric_table = (
    network_metrics.copy()
)

network_metric_table[
    "method"
] = network_metric_table[
    "method_id"
].map(
    method_names
)

network_metric_table[
    [
        "network_id",
        "metric",
        "method",
        "value",
    ]
].sort_values(
    [
        "metric",
        "network_id",
        "method",
    ]
).reset_index(
    drop=True
)